### **[Fund Flow Strategy](https://medium.com/coding-nexus/bc62cbd6165c)**

> *A Python Powered ETF Trading Strategy and With Full Setup*

In [1]:
!pip install -qq yfinance vectorbt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.7/451.7 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 68.2 MB/s eta 0:00:00


In [2]:
import sys

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import yfinance as yf
import vectorbt as vbt

from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules

np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)

pd.set_option('display.precision', 4)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', '{:0.4f}'.format)

%autosave 15

Autosaving every 15 seconds


In [3]:
start_date = "2010-01-01"
end_date = pd.Timestamp.today().strftime("%Y-%m-%d")  # Up to current date (e.g., 2025-08-24)
symbols = ["TLT", "SPY"]

# Download closing prices; handle errors
try:
  price_data = yf.download(symbols, start=start_date, end=end_date, auto_adjust=False, progress=False)["Close"]
except Exception as e:
  raise Exception(f"Data download failed: {e}")

# Forward-fill missing data for robustness (e.g., holidays)
price_data = price_data.fillna(method='ffill').dropna()

display(price_data.head(15))

# Group by month to find first/last trading days
dates = price_data.index
year_month = dates.to_period('M')
grouped = pd.DataFrame({"dt": dates, "ym": year_month}).groupby("ym")["dt"]
first_trading_days = grouped.first().values
last_trading_days = grouped.last().values

Ticker,SPY,TLT
Date,,
2010-01-04,113.3300,89.8100
2010-01-05,113.6300,90.3900
2010-01-06,113.7100,89.1800
2010-01-07,114.1900,89.3300
2010-01-08,114.5700,89.2900
2010-01-11,114.7300,88.8000
2010-01-12,113.6600,90.3200
2010-01-13,114.6200,89.2700
2010-01-14,114.9300,90.5200


#### **Create Trading Signal Events**

In [4]:
# Find previous trading days (e.g., 7 before month-end)
def get_prev_trading_day_idx(trading_dates, base_dates, offset):
  idx = []
  trading_dates = pd.Series(trading_dates)  # For efficient searching
  for d in base_dates:
    pos = trading_dates.searchsorted(d)  # Insertion point
    prev_idx = pos - offset
    if prev_idx >= 0:
      idx.append(trading_dates.iloc[prev_idx])
  return pd.DatetimeIndex(idx)

# Find offset trading days (e.g., 1 after)
def get_offset_trading_day_idx(trading_dates, base_dates, offset):
  idx = []
  trading_dates = pd.Series(trading_dates)
  for d in base_dates:
    pos = trading_dates.searchsorted(d)
    target_idx = pos + offset
    if target_idx < len(trading_dates):
      idx.append(trading_dates.iloc[target_idx])
  return pd.DatetimeIndex(idx)

# Key dates
pre_end_idx = get_prev_trading_day_idx(dates, last_trading_days, 7)  # 7 days before end
month_start_idx = get_offset_trading_day_idx(dates, last_trading_days, 1)  # Month start
week_after_start_idx = get_offset_trading_day_idx(dates, month_start_idx, 7)  # Week later

#### **Define and Organize Trading Signals**

In [5]:
# Initialize signals DataFrame for simplicity
signals = pd.DataFrame(
  index=dates,
  columns=pd.MultiIndex.from_product([symbols, ['long_entry', 'long_exit', 'short_entry', 'short_exit']]),
  data=False
)

# trade_filter = price_data['TLT'].pct_change(20) > price_data['SPY'].pct_change(20)

# Entries: Long TLT/short SPY before end; reverse at start
signals.loc[pre_end_idx, ("TLT", "long_entry")] = True
signals.loc[pre_end_idx, ("SPY", "short_entry")] = True
signals.loc[month_start_idx, ("TLT", "short_entry")] = True
signals.loc[month_start_idx, ("SPY", "long_entry")] = True

# Exits: Close at start for first leg; week later for second
signals.loc[month_start_idx, ("TLT", "long_exit")] = True
signals.loc[month_start_idx, ("SPY", "short_exit")] = True
signals.loc[week_after_start_idx, ("TLT", "short_exit")] = True
signals.loc[week_after_start_idx, ("SPY", "long_exit")] = True

# Format for vectorbt
long_entry = pd.DataFrame({sym: signals[(sym, "long_entry")] for sym in symbols})
long_exit = pd.DataFrame({sym: signals[(sym, "long_exit")] for sym in symbols})
short_entry = pd.DataFrame({sym: signals[(sym, "short_entry")] for sym in symbols})
short_exit = pd.DataFrame({sym: signals[(sym, "short_exit")] for sym in symbols})

#### **Backtest the Trading Strategy**

In [6]:
# Backtest with $100,000 initial cash
pf = vbt.Portfolio.from_signals(
  price_data,
  entries=long_entry,
  exits=long_exit,
  short_entries=short_entry,
  short_exits=short_exit,
  freq='D',
  size_type=1,  # Percentage of cash
  size=np.inf,  # Unlimited (for simplicity; adjust to 1 for fixed shares)
  init_cash=100_000
)

# Display stats
display(pf.stats())

,agg_func_mean
Start,2010-01-04 00:00:00
End,2026-05-29 00:00:00
Period,4126 days 00:00:00
Start Value,100000.0000
End Value,44683.8374
Total Return [%],-55.3162
Benchmark Return [%],281.4962
Max Gross Exposure [%],100.0000
Total Fees Paid,0.0000
Max Drawdown [%],65.7740
